In [12]:
from google.colab import files

uploaded = files.upload()

zip_path = next(iter(uploaded.keys()))
print("Uploaded file:", zip_path)

Saving archive (1).zip to archive (1) (4).zip
Uploaded file: archive (1) (4).zip


In [13]:
import zipfile
import os

extract_path = "/content/conversation_data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

for root, dirs, filenames in os.walk(extract_path):
    for filename in filenames:
        print(os.path.join(root, filename))

/content/conversation_data/stats.json
/content/conversation_data/README.md
/content/conversation_data/data/release/v1/data/ami/notes/train.parquet
/content/conversation_data/data/release/v1/data/ami/notes/validation.parquet
/content/conversation_data/data/release/v1/data/ami/notes/test.parquet
/content/conversation_data/data/release/v1/data/ami/sentences/train.parquet
/content/conversation_data/data/release/v1/data/ami/sentences/validation.parquet
/content/conversation_data/data/release/v1/data/ami/sentences/test.parquet
/content/conversation_data/data/release/v1/data/synthetic/notes/train.parquet
/content/conversation_data/data/release/v1/data/synthetic/notes/validation.parquet
/content/conversation_data/data/release/v1/data/synthetic/notes/test.parquet
/content/conversation_data/data/release/v1/data/synthetic/sentences/train.parquet
/content/conversation_data/data/release/v1/data/synthetic/sentences/validation.parquet
/content/conversation_data/data/release/v1/data/synthetic/sentence

In [14]:

!pip -q install pyarrow

import pandas as pd
import os

base_path = "/content/conversation_data/data/release/v1/data"

files_to_check = [
    f"{base_path}/ami/notes/train.parquet",
    f"{base_path}/ami/sentences/train.parquet",
    f"{base_path}/synthetic/notes/train.parquet",
    f"{base_path}/synthetic/sentences/train.parquet",
]

for file_path in files_to_check:
    print("\n" + "=" * 60)
    print("FILE:", file_path)

    df = pd.read_parquet(file_path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(2))



FILE: /content/conversation_data/data/release/v1/data/ami/notes/train.parquet
Shape: (97, 11)
Columns: ['note_id', 'source', 'meeting_type', 'split', 'n_sentences', 'n_words', 'n_participants', 'summary_abstract', 'summary_decisions', 'summary_actions', 'summary_problems']


,note_id,source,meeting_type,split,n_sentences,n_words,n_participants,summary_abstract,summary_decisions,summary_actions,summary_problems
0,ES2002a,ami,ami_design,train,434,2601,4,The project manager introduced the upcoming pr...,The remote will sell for 25 Euro. The remote w...,The industrial designer will work on the worki...,Whether the remote will be used exclusively fo...
1,ES2002b,ami,ami_design,train,976,6994,4,The project manager briefed the team on some n...,The team will not work with teletext. The remo...,The industrial designer will work on the compo...,How to categorize the functions on the remote....



FILE: /content/conversation_data/data/release/v1/data/ami/sentences/train.parquet
Shape: (75996, 17)
Columns: ['note_id', 'source', 'meeting_type', 'domain', 'split', 'sentence_id', 'speaker', 'sentence_text', 'sentence_text_asr', 'importance', 'confidence', 'zone', 'segment_id', 'family_id', 'hard_negative', 'token_count', 'spans']


,note_id,source,meeting_type,domain,split,sentence_id,speaker,sentence_text,sentence_text_asr,importance,confidence,zone,segment_id,family_id,hard_negative,token_count,spans
0,ES2002a,ami,ami_design,product_design,train,0,B,Okay,okay,0,0.8,discussion,None,None,False,1,[]
1,ES2002a,ami,ami_design,product_design,train,1,B,Right,right,0,0.8,discussion,None,None,False,1,[]



FILE: /content/conversation_data/data/release/v1/data/synthetic/notes/train.parquet
Shape: (1200, 11)
Columns: ['note_id', 'source', 'meeting_type', 'split', 'n_sentences', 'n_words', 'n_participants', 'summary_abstract', 'summary_decisions', 'summary_actions', 'summary_problems']


,note_id,source,meeting_type,split,n_sentences,n_words,n_participants,summary_abstract,summary_decisions,summary_actions,summary_problems
0,syn-00002,synthetic,quick_sync,train,28,195,2,None,None,None,None
1,syn-00003,synthetic,quick_sync,train,39,226,2,None,None,None,None



FILE: /content/conversation_data/data/release/v1/data/synthetic/sentences/train.parquet
Shape: (110230, 17)
Columns: ['note_id', 'source', 'meeting_type', 'domain', 'split', 'sentence_id', 'speaker', 'sentence_text', 'sentence_text_asr', 'importance', 'confidence', 'zone', 'segment_id', 'family_id', 'hard_negative', 'token_count', 'spans']


,note_id,source,meeting_type,domain,split,sentence_id,speaker,sentence_text,sentence_text_asr,importance,confidence,zone,segment_id,family_id,hard_negative,token_count,spans
0,syn-00002,synthetic,quick_sync,customer_support,train,0,Tafadzwa,It's freezing here today.,it's freezing here today,0,1.0,discussion,fill-029,fill-029,False,4,[]
1,syn-00002,synthetic,quick_sync,customer_support,train,1,Tafadzwa,Thanks for picking up.,thanks for picking up,0,1.0,context,ctx-quick_sync-sup-001-v1,ctx-quick_sync-sup-001,False,4,[]


In [15]:
!pip -q install transformers sentencepiece rouge_score

import pandas as pd
import os

base = "/content/conversation_data/data/release/v1/data/ami"

def load_split(split):
    notes = pd.read_parquet(f"{base}/notes/{split}.parquet")
    sentences = pd.read_parquet(f"{base}/sentences/{split}.parquet")

    # Keep the sentences in their original order
    transcripts = (
        sentences.sort_values(["note_id", "sentence_id"])
        .groupby("note_id")["sentence_text"]
        .apply(lambda x: " ".join(x.dropna().astype(str)))
        .rename("transcript")
        .reset_index()
    )

    data = notes.merge(transcripts, on="note_id", how="inner")

    # Use the meeting overview as the reference summary
    data = data.dropna(subset=["summary_abstract"])
    data = data[
        data["summary_abstract"].astype(str).str.strip().ne("")
    ]

    return data

train_df = load_split("train")
val_df = load_split("validation")
test_df = load_split("test")

print("Training meetings:", len(train_df))
print("Validation meetings:", len(val_df))
print("Test meetings:", len(test_df))

display(train_df[["note_id", "transcript", "summary_abstract"]].head(2))

In [16]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "sshleifer/distilbart-cnn-12-6"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded on: cpu


In [17]:

import time
import torch

sample = test_df.iloc[0]
text = sample["transcript"][:2000]

start = time.time()

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        num_beams=1,
        do_sample=False
    )

summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("SUMMARY:\n", summary)
print(f"\nTime taken: {time.time() - start:.1f} seconds")


[transformers] Both `max_new_tokens` (=80) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SUMMARY:
  Meetings are the first meeting of the project manager's team . They are working on a remote control remote control . They will do some stuff, Get to know each other a bit better to feel more comfortable with each other . Then we'll go do tool training, Talk about the project plan, Discuss our own ideas and everything .

Time taken: 13.7 seconds


In [18]:

import os
import pandas as pd

base = "/content/conversation_data/data/release/v1/data/ami"

notes = pd.read_parquet(f"{base}/notes/test.parquet")
sentences = pd.read_parquet(f"{base}/sentences/test.parquet")

transcripts = (
    sentences.sort_values(["note_id", "sentence_id"])
    .groupby("note_id")["sentence_text"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .rename("transcript")
    .reset_index()
)

test_df = notes.merge(transcripts, on="note_id", how="inner")
test_df = test_df.dropna(subset=["summary_abstract"])
test_df = test_df[
    test_df["summary_abstract"].astype(str).str.strip().ne("")
]

print("Test dataset ready:", len(test_df), "meetings")


Test dataset ready: 20 meetings


In [19]:

from rouge_score import rouge_scorer
import time

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

scores_list = []
start = time.time()

for i in range(min(3, len(test_df))):
    row = test_df.iloc[i]
    text = row["transcript"][:2000]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=1,
            do_sample=False
        )

    generated = tokenizer.decode(
        output[0], skip_special_tokens=True
    )

    reference = row["summary_abstract"]
    scores = scorer.score(reference, generated)

    scores_list.append({
        "Meeting": row["note_id"],
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure
    })

    print(f"\nMeeting: {row['note_id']}")
    print("Generated:", generated)

results = pd.DataFrame(scores_list)
print("\nEvaluation results:")
display(results.round(3))

print("\nAverage ROUGE scores:")
display(results[["ROUGE-1", "ROUGE-2", "ROUGE-L"]].mean().round(3))

print(f"\nTotal time: {time.time() - start:.1f} seconds")


[transformers] Both `max_new_tokens` (=80) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Meeting: ES2004a
Generated:  Meetings are the first meeting of the project manager's team . They are working on a remote control remote control . They will do some stuff, Get to know each other a bit better to feel more comfortable with each other . Then we'll go do tool training, Talk about the project plan, Discuss our own ideas and everything .


[transformers] Both `max_new_tokens` (=80) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Meeting: ES2004b
Generated:  The company's budget would have to affect the product, not to reflect our budget, so that we might have a bit of dissonance between what our budget was and what we want it to look like . The company wanted to reflect the company's image in their product, we put fashion in electronics, you know that kind of s thing .


[transformers] Both `max_new_tokens` (=80) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Meeting: ES2004c
Generated:  This is where we talk about properties, materials, user-interface and trend-watching . We decided on a leaving out the voice recognition, We've decided on there being a flip design And um a different shape from what's normal . We were thinking a shell, but something along those lines, Just a different . shape from . what's normally normal .

Evaluation results:


,Meeting,ROUGE-1,ROUGE-2,ROUGE-L
0,ES2004a,0.309,0.065,0.181
1,ES2004b,0.204,0.024,0.118
2,ES2004c,0.183,0.024,0.096



Average ROUGE scores:


,0
ROUGE-1,0.232
ROUGE-2,0.037
ROUGE-L,0.131



Total time: 41.8 seconds


In [20]:

!pip -q install -U transformers sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.0 MB/s eta 0:00:00


In [21]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "pszemraj/led-base-book-summary"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)


config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  648MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/296 [00:00<?, ?it/s]

Model loaded on: cpu


In [22]:
sample = test_df.iloc[0]
transcript = sample["transcript"]

inputs = tokenizer(
    transcript,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=120,
        num_beams=2,
        do_sample=False
    )

generated = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("GENERATED SUMMARY:\n")
print(generated)

print("\nREFERENCE SUMMARY:\n")
print(sample["summary_abstract"])

[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


GENERATED SUMMARY:

The Project Manager introduces herself as Sarah, The Project Manager. She explains that she is the project manager and that her first meeting was her first "first meeting," surprisingly enough. Her first meeting consisted of getting to know each other a bit better so they could feel more comfortable with each other. This included: design, conceptual design, What we're thinking, How it's gonna go And then the detailed design, How we're actually gonna put it into practice and make it work" . In this case, though, these are the three unique features she thinks belong to a dog. They'll have

REFERENCE SUMMARY:

The Project Manager gave an introduction to the goal of the project, to create a trendy yet user-friendly remote. She presented a long-range agenda for the whole project. The group introduced themselves to each other and practiced with the meeting room tools by drawing on the board. The Project Manager presented the project budget, the projected price point, and 

In [23]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

scores = scorer.score(
    sample["summary_abstract"],
    generated
)

for name, result in scores.items():
    print(f"{name}: {result.fmeasure:.4f}")

rouge1: 0.3090
rouge2: 0.0606
rougeL: 0.1803


In [24]:
# Keep the LED model and tokenizer for comparison
led_model = model
led_tokenizer = tokenizer

# Load DistilBART
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

distil_name = "sshleifer/distilbart-cnn-12-6"
distil_tokenizer = AutoTokenizer.from_pretrained(distil_name)
distil_model = AutoModelForSeq2SeqLM.from_pretrained(distil_name)

print("Both models are ready to compare.")

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Both models are ready to compare.


In [25]:
from rouge_score import rouge_scorer
import pandas as pd
import torch

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def generate_summary(text, model, tokenizer, max_input=1024):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input
    ).to(next(model.parameters()).device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            num_beams=2,
            do_sample=False
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

results = []

for i in range(min(3, len(test_df))):
    transcript = test_df.iloc[i]["transcript"]
    reference = test_df.iloc[i]["summary_abstract"]

    led_summary = generate_summary(
        transcript, led_model, led_tokenizer
    )
    distil_summary = generate_summary(
        transcript, distil_model, distil_tokenizer
    )

    led_scores = scorer.score(reference, led_summary)
    distil_scores = scorer.score(reference, distil_summary)

    results.append({
        "Meeting": test_df.iloc[i]["note_id"],
        "LED ROUGE-1": led_scores["rouge1"].fmeasure,
        "DistilBART ROUGE-1": distil_scores["rouge1"].fmeasure,
        "LED ROUGE-2": led_scores["rouge2"].fmeasure,
        "DistilBART ROUGE-2": distil_scores["rouge2"].fmeasure,
        "LED ROUGE-L": led_scores["rougeL"].fmeasure,
        "DistilBART ROUGE-L": distil_scores["rougeL"].fmeasure,
    })

comparison = pd.DataFrame(results)
display(comparison.round(4))

print("\nAverage scores across the 3 meetings:")
display(comparison.drop(columns="Meeting").mean().round(4).to_frame("Average"))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingfa

,Meeting,LED ROUGE-1,DistilBART ROUGE-1,LED ROUGE-2,DistilBART ROUGE-2,LED ROUGE-L,DistilBART ROUGE-L
0,ES2004a,0.3090,0.2798,0.0606,0.0419,0.1803,0.1762
1,ES2004b,0.2207,0.2366,0.0347,0.0538,0.1379,0.1374
2,ES2004c,0.2337,0.2429,0.0346,0.0571,0.1375,0.1296



Average scores across the 3 meetings:


,Average
LED ROUGE-1,0.2545
DistilBART ROUGE-1,0.2531
LED ROUGE-2,0.0433
DistilBART ROUGE-2,0.0510
LED ROUGE-L,0.1519
DistilBART ROUGE-L,0.1477


In [26]:
results_all = []

for i in range(len(test_df)):
    transcript = test_df.iloc[i]["transcript"]
    reference = test_df.iloc[i]["summary_abstract"]

    led_summary = generate_summary(
        transcript, led_model, led_tokenizer
    )
    distil_summary = generate_summary(
        transcript, distil_model, distil_tokenizer
    )

    led_scores = scorer.score(reference, led_summary)
    distil_scores = scorer.score(reference, distil_summary)

    results_all.append({
        "Meeting": test_df.iloc[i]["note_id"],
        "LED ROUGE-1": led_scores["rouge1"].fmeasure,
        "DistilBART ROUGE-1": distil_scores["rouge1"].fmeasure,
        "LED ROUGE-2": led_scores["rouge2"].fmeasure,
        "DistilBART ROUGE-2": distil_scores["rouge2"].fmeasure,
        "LED ROUGE-L": led_scores["rougeL"].fmeasure,
        "DistilBART ROUGE-L": distil_scores["rougeL"].fmeasure,
    })

    print(f"Completed meeting {i + 1}/{len(test_df)}")

results_df = pd.DataFrame(results_all)

print("\nAVERAGE ROUGE SCORES — ALL TEST MEETINGS")
display(results_df.drop(columns="Meeting").mean().round(4).to_frame("Average"))

print("\nINDIVIDUAL MEETING SCORES")
display(results_df.round(4))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed meeting 1/20


[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed meeting 2/20


[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=120) and `max_length`(=1024) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Completed meeting 20/20

AVERAGE ROUGE SCORES — ALL TEST MEETINGS


,Average
LED ROUGE-1,0.2621
DistilBART ROUGE-1,0.2265
LED ROUGE-2,0.0505
DistilBART ROUGE-2,0.0451
LED ROUGE-L,0.1404
DistilBART ROUGE-L,0.1402



INDIVIDUAL MEETING SCORES


,Meeting,LED ROUGE-1,DistilBART ROUGE-1,LED ROUGE-2,DistilBART ROUGE-2,LED ROUGE-L,DistilBART ROUGE-L
0,ES2004a,0.3090,0.2798,0.0606,0.0419,0.1803,0.1762
1,ES2004b,0.2207,0.2366,0.0347,0.0538,0.1379,0.1374
2,ES2004c,0.2337,0.2429,0.0346,0.0571,0.1375,0.1296
3,ES2004d,0.2756,0.1611,0.0476,0.0000,0.1339,0.1232
4,ES2014a,0.3091,0.3125,0.0826,0.0811,0.1636,0.1875
5,ES2014b,0.2041,0.1040,0.0342,0.0081,0.1088,0.0720
6,ES2014c,0.2533,0.1864,0.0805,0.0578,0.1600,0.1219
7,ES2014d,0.1622,0.2892,0.0136,0.0405,0.0878,0.1446
8,IS1009a,0.4330,0.2222,0.1107,0.0498,0.2062,0.1399
9,IS1009b,0.2699,0.1953,0.0279,0.0157,0.1315,0.1094


In [27]:
# Select a meeting from the test dataset
sample = test_df.iloc[0]

print("=" * 50)
print("MEETING:", sample["note_id"])
print("=" * 50)

print("\n📝 MEETING SUMMARY\n")
print(sample["summary_abstract"])

print("\n✅ DECISIONS\n")
print(sample["summary_decisions"])

print("\n📌 ACTION ITEMS\n")
print(sample["summary_actions"])

print("\n⚠️ PROBLEMS DISCUSSED\n")
print(sample["summary_problems"])

MEETING: ES2004a

📝 MEETING SUMMARY

The Project Manager gave an introduction to the goal of the project, to create a trendy yet user-friendly remote. She presented a long-range agenda for the whole project. The group introduced themselves to each other and practiced with the meeting room tools by drawing on the board. The Project Manager presented the project budget, the projected price point, and the projected profit aim for the project. Then the group began a discussion about their own experiences with remote controls to generate initial design ideas for making the product user-friendly. They discussed grouping features into a menu and adding an LCD display. They also discussed the look of various materials that may be used in the design, in keeping with the company's goal to create fashionable electronics.

✅ DECISIONS

NA.

📌 ACTION ITEMS

The group will prepare for the functional design meeting to discuss the components and functions of the product.

⚠️ PROBLEMS DISCUSSED

There 

In [28]:
!pip -q install gradio

In [29]:
import gradio as gr
import torch

def meeting_summarizer(transcript):
    if not transcript or not transcript.strip():
        return "Please enter a meeting transcript."

    try:
        summary = generate_summary(
            transcript,
            led_model,
            led_tokenizer
        )

        return summary

    except Exception as e:
        return f"Error generating summary: {str(e)}"


demo = gr.Interface(
    fn=meeting_summarizer,
    inputs=gr.Textbox(
        lines=12,
        label="Meeting Transcript",
        placeholder="Paste your meeting transcript here..."
    ),
    outputs=gr.Textbox(
        lines=6,
        label="AI-Generated Summary"
    ),
    title="AI Meeting Summarizer",
    description=(
        "Generate concise summaries from meeting transcripts "
        "using a transformer-based language model."
    ),
    examples=[
        [
            "The team discussed the project deadline. "
            "Riya will prepare the design by Friday. "
            "Rahul will test the application next week. "
            "The team agreed to review progress on Monday."
        ]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2bc99ba9f7fcbbd494.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [30]:
import gradio as gr
import re
import torch

def meeting_summarizer(transcript):
    if not transcript or not transcript.strip():
        return "Please enter a transcript.", "", ""

    # Generate the AI summary
    summary = generate_summary(
        transcript,
        led_model,
        led_tokenizer
    )

    # Split transcript into sentences
    sentences = re.split(r'(?<=[.!?])\s+', transcript.strip())

    # Basic prototype: identify sentences containing relevant phrases
    decision_words = [
        "decided", "agreed", "decision", "approved",
        "will proceed", "finalized"
    ]

    action_words = [
        "will", "must", "should", "action item",
        "responsible", "deadline", "by tomorrow",
        "by friday", "by monday"
    ]

    decisions = [
        s for s in sentences
        if any(word in s.lower() for word in decision_words)
    ]

    actions = [
        s for s in sentences
        if any(word in s.lower() for word in action_words)
    ]

    decisions_text = "\n".join(
        f"• {s}" for s in decisions
    ) or "No clear decisions detected."

    actions_text = "\n".join(
        f"• {s}" for s in actions
    ) or "No clear action items detected."

    return summary, decisions_text, actions_text


with gr.Blocks(title="AI Meeting Summarizer") as demo:
    gr.Markdown("# 📝 AI Meeting Summarizer")
    gr.Markdown(
        "Generate meeting summaries, identify decisions, "
        "and find possible action items."
    )

    transcript_input = gr.Textbox(
        lines=12,
        label="Meeting Transcript",
        placeholder="Paste your meeting transcript here..."
    )

    submit_btn = gr.Button("Summarize Meeting", variant="primary")

    summary_output = gr.Textbox(
        lines=6, label="AI-Generated Summary"
    )
    decisions_output = gr.Textbox(
        lines=5, label="Possible Decisions"
    )
    actions_output = gr.Textbox(
        lines=5, label="Possible Action Items"
    )

    submit_btn.click(
        fn=meeting_summarizer,
        inputs=transcript_input,
        outputs=[
            summary_output,
            decisions_output,
            actions_output
        ]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6406ef16fb9bb9310d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:

import gradio as gr
import os
import tempfile

def create_report(transcript):
    summary, decisions, actions = meeting_summarizer(transcript)

    report = f"""AI MEETING SUMMARIZER REPORT
{'=' * 35}

MEETING SUMMARY
---------------
{summary}

KEY DECISIONS
-------------
{decisions}

ACTION ITEMS
------------
{actions}

SOURCE TRANSCRIPT
-----------------
{transcript}
"""

    report_path = os.path.join(
        tempfile.gettempdir(),
        "meeting_summary_report.txt"
    )

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report)

    return summary, decisions, actions, report_path


with gr.Blocks(title="AI Meeting Summarizer") as demo:
    gr.Markdown("# 📝 AI Meeting Summarizer")
    gr.Markdown(
        "Summarize meetings, identify decisions, "
        "and download your report."
    )

    transcript_input = gr.Textbox(
        lines=12,
        label="Meeting Transcript",
        placeholder="Paste your meeting transcript here..."
    )

    submit_btn = gr.Button(
        "Summarize Meeting",
        variant="primary"
    )

    summary_output = gr.Textbox(
        lines=6, label="AI-Generated Summary"
    )
    decisions_output = gr.Textbox(
        lines=5, label="Possible Decisions"
    )
    actions_output = gr.Textbox(
        lines=5, label="Possible Action Items"
    )

    download_output = gr.File(label="Download Your Report")

    submit_btn.click(
        fn=create_report,
        inputs=transcript_input,
        outputs=[
            summary_output,
            decisions_output,
            actions_output,
            download_output
        ]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://07fc33c47baba2636c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:

import re

def extract_action_items(transcript):
    sentences = re.split(r'(?<=[.!?])\s+', transcript.strip())
    action_items = []

    task_pattern = re.compile(
        r"\b(?:will|must|needs to|need to|should|has to|have to|"
        r"is responsible for)\b",
        re.IGNORECASE
    )

    deadline_pattern = re.compile(
        r"\b(?:by|before|until|on)\s+"
        r"(?:next\s+)?"
        r"(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|"
        r"tomorrow|today|tonight|next week|the end of the week|"
        r"\d{1,2}(?:st|nd|rd|th)?(?:\s+of\s+\w+)?|\w+\s+\d{1,2})\b",
        re.IGNORECASE
    )

    for sentence in sentences:
        if not task_pattern.search(sentence):
            continue

        deadline_match = deadline_pattern.search(sentence)
        deadline = deadline_match.group(0) if deadline_match else "Not specified"

        # Simple heuristic: use the first word as a possible speaker/owner
        owner_match = re.match(r"^\s*([A-Z][a-z]+)\s+", sentence)
        owner = owner_match.group(1) if owner_match else "Owner not identified"

        action_items.append({
            "Task": sentence.strip(),
            "Possible owner": owner,
            "Deadline": deadline
        })

    return action_items


# Quick test
sample_transcript = """
Riya will finish the user interface by Friday.
Rahul must test the login system before Monday.
The team discussed the project budget.
"""

for item in extract_action_items(sample_transcript):
    print(item)


{'Task': 'Riya will finish the user interface by Friday.', 'Possible owner': 'Riya', 'Deadline': 'by Friday'}
{'Task': 'Rahul must test the login system before Monday.', 'Possible owner': 'Rahul', 'Deadline': 'before Monday'}


In [33]:

    action_items = extract_action_items(transcript)

    actions_text = "\n".join(
        f"• Task: {item['Task']}\n"
        f"  Possible owner: {item['Possible owner']}\n"
        f"  Deadline: {item['Deadline']}"
        for item in action_items
    ) or "No possible action items detected."


## Conclusion

The AI Meeting Summarizer is a Generative AI project that uses pretrained transformer models to generate concise summaries from meeting transcripts. It compares LED and DistilBART using ROUGE evaluation metrics.

The application includes:
- Automatic meeting summarization
- Identification of possible decisions
- Extraction of potential action items, owners, and deadlines
- A Gradio web interface
- Downloadable meeting reports

The project demonstrates how pretrained NLP models can be integrated into a practical application. Action-item extraction currently uses rule-based techniques, so results may require manual verification.

**Future Improvements:** Improve action-item extraction, evaluate models on a larger dataset, and deploy the application for reliable long-term access.